In [1]:
!pip install -q transformers accelerate sentencepiece


In [2]:
import os
import ast
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

torch.manual_seed(42)


Using device: cuda


In [4]:
NBME_DIR = "/kaggle/input/nbme-score-clinical-patient-notes"

patient_notes = pd.read_csv(f"{NBME_DIR}/patient_notes.csv")
train = pd.read_csv(f"{NBME_DIR}/train.csv")

print("Patient notes:", patient_notes.shape)
print("Annotations:", train.shape)


Patient notes: (42146, 3)
Annotations: (14300, 6)


In [5]:
annotated_pn_nums = train["pn_num"].unique()

annotated_notes = patient_notes[
    patient_notes["pn_num"].isin(annotated_pn_nums)
][["pn_num", "pn_history"]].reset_index(drop=True)

print("Annotated notes:", annotated_notes.shape)
annotated_notes.head()


Annotated notes: (1000, 2)


,pn_num,pn_history
0,16,HPI: 17yo M presents with palpitations. Patien...
1,41,17 Y/O M CAME TO THE CLINIC C/O HEART POUNDING...
2,46,Mr. Cleveland is a 17yo M who was consented by...
3,82,17 yo M w/ no cardiac or arrhythmia PMH presen...
4,100,HPI: Dillon Cleveland is an otherwise healthy ...


In [6]:
ANNOT_PATH = "/kaggle/working/nbme_annotated_notes.csv"
annotated_notes.to_csv(ANNOT_PATH, index=False)

assert os.path.exists(ANNOT_PATH)


In [7]:
MODEL_NAME = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
model.eval()


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-29 09:39:19.125243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769679559.439726      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769679559.529563      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769679560.301737      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769679560.301785      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769679560.301788      23

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [8]:
MAX_CHARS = 1200

def truncate(text):
    return text[:MAX_CHARS] if isinstance(text, str) else ""

def build_prompt(note):
    return (
        "You are a clinical information extraction system.\n\n"
        "Task: Extract patient-reported symptoms or clinician-observed findings "
        "from the clinical note below.\n\n"
        "Rules:\n"
        "- Extract symptoms or findings only\n"
        "- Exclude diagnoses, medications, procedures, labs, demographics\n"
        "- Exclude explicitly negated symptoms\n"
        "- Do NOT infer new symptoms\n"
        "- Output one symptom per line starting with '-'\n"
        "- If no symptoms are present, output exactly: none\n\n"
        "Clinical note:\n"
        f"{truncate(note)}\n\n"
        "Symptoms:\n"
    )


In [9]:
def parse_symptoms(text):
    if "Symptoms:" in text:
        text = text.split("Symptoms:", 1)[1]

    lines = text.split("\n")
    symptoms = []

    for line in lines:
        line = line.strip()
        if line.startswith("-"):
            s = line[1:].strip().lower()
            if len(s) < 2:
                continue
            if any(x in s for x in [
                "extract", "exclude", "rules", "output", "clinical note"
            ]):
                continue
            symptoms.append(s)

    if len(symptoms) == 0:
        return ["none"]

    return list(dict.fromkeys(symptoms))


In [10]:
BATCH_SIZE = 2
results = []

for i in tqdm(range(0, len(annotated_notes), BATCH_SIZE)):
    batch = annotated_notes.iloc[i:i+BATCH_SIZE]

    prompts = [build_prompt(n) for n in batch["pn_history"]]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # ✅ decode GENERATED TOKENS ONLY
    decoded = []
    for j, output in enumerate(outputs):
        input_len = inputs["input_ids"][j].shape[0]
        gen_tokens = output[input_len:]
        decoded.append(tokenizer.decode(gen_tokens, skip_special_tokens=True))

    for pn_num, text in zip(batch["pn_num"], decoded):
        results.append({
            "pn_num": pn_num,
            "model": "biomistral-7b",
            "predicted_symptoms": parse_symptoms(text)
        })

    torch.cuda.empty_cache()


  0%|          | 0/500 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
100%|██████████| 500/500 [54:53<00:00,  6.59s/it]


In [11]:
OUT_DIR = "/kaggle/working/nbme_biomistral_annotated"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PATH = f"{OUT_DIR}/biomistral_predictions_annotated_ff.csv"

df = pd.DataFrame(results)
df.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
df.head()


Saved: /kaggle/working/nbme_biomistral_annotated/biomistral_predictions_annotated_ff.csv


,pn_num,model,predicted_symptoms
0,16,biomistral-7b,[palpitations]
1,41,biomistral-7b,[none]
2,46,biomistral-7b,[palpitations]
3,82,biomistral-7b,[none]
4,100,biomistral-7b,[light headedness]


In [12]:
df.sample(5)


,pn_num,model,predicted_symptoms
484,44390,biomistral-7b,[none]
865,82817,biomistral-7b,[none]
867,82848,biomistral-7b,[sleep disturbance]
917,91251,biomistral-7b,[headache]
882,83757,biomistral-7b,[sleep disturbance]
